In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

RESULTS_DIR = "../results/accuracy"
PLOTS_DIR   = "../plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Load accuracy JSONs
# Each file has scores for all 5 benchmarks
def load_eval(path):
    with open(path) as f:
        data = json.load(f)
    r = data["results"]
    return {
        "mmlu":       r["mmlu"]["acc"],
        "hellaswag":  r["hellaswag"]["acc_norm"],
        "gsm8k":      r["gsm8k"]["acc"],
        "humaneval":  r["humaneval"]["pass@1"],
        "truthfulqa": r["truthfulqa_mc1"]["acc"],
    }

scores = {
    "fp16": load_eval(f"{RESULTS_DIR}/fp16_eval.json"),
    "int8": load_eval(f"{RESULTS_DIR}/int8_eval.json"),
    "gptq": load_eval(f"{RESULTS_DIR}/gptq_eval.json"),
    "awq":  load_eval(f"{RESULTS_DIR}/awq_eval.json"),
}

accuracy_df = pd.DataFrame(scores).T
accuracy_df.index.name = "model"
print(accuracy_df.round(3))

In [ ]:
# Accuracy table
# Clean formatted table showing all scores
print("\nAccuracy Scores (higher is better):")
print(accuracy_df.round(3).to_string())

In [ ]:
# Accuracy heatmap
# Color coded — green = close to FP16, red = degraded
# Normalize each column relative to FP16 baseline
fp16_scores  = accuracy_df.loc["fp16"]
degradation  = accuracy_df.div(fp16_scores) # ratio vs FP16

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    degradation,
    annot=accuracy_df.round(3),  # show actual scores in cells
    fmt="",
    cmap="RdYlGn",
    vmin=0.90,
    vmax=1.0,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Accuracy Heatmap (color = ratio vs FP16 baseline)")
ax.set_xlabel("Benchmark")
ax.set_ylabel("Model Variant")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/accuracy_heatmap.png", dpi=150)
plt.show()
print("Saved accuracy_heatmap.png")

In [ ]:
# Degradation table
# How many points each variant lost vs FP16
degradation_pts = accuracy_df.subtract(fp16_scores)
print("\nAccuracy Degradation vs FP16 (negative = worse):")
print(degradation_pts.round(3).to_string())

In [ ]:
# Per benchmark bar chart
# Side by side bars for all models, one group per benchmark
fig, ax = plt.subplots(figsize=(12, 6))
accuracy_df.T.plot(kind="bar", ax=ax, width=0.7)
ax.set_ylabel("Accuracy Score")
ax.set_title("Accuracy by Benchmark and Model Variant")
ax.set_xticklabels(accuracy_df.columns, rotation=30)
ax.legend(title="Model")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/accuracy_by_benchmark.png", dpi=150)
plt.show()
print("Saved accuracy_by_benchmark.png")